# MEDHA — Qwen2.5-3B Explainer Training (FINAL) [UPDATED: 2026-06-11 12:48:48]

**Key design decisions:**
- First 250 examples (validated BD exam questions) → always in training, never in validation
- MAX_SEQ_LEN reduced to 256 (actual max is 208 tokens — saves 30% GPU memory)
- Epochs reduced to 6 (15 was massive overkill for 1,158 examples)
- LR lowered to 1e-4 (more stable for 3B model)
- Class imbalance handled via weighted sampling
- Single merge/push — no double-merge crash
- Hackathon mode: auto-saves checkpoint every epoch

**Expected: ~1.5 hours on T4 x2**

In [ ]:
# ── CELL 1: Install & Environment Setup ────────────────────────────────────
import os
import subprocess
import sys
import importlib

print("Installing packages...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 
                'transformers', 'datasets', 'peft', 'bitsandbytes', 'accelerate', 'trl'])

# Invalidate python's module cache so it finds the newly installed packages WITHOUT restarting
importlib.invalidate_caches()
import site
site.main()

import warnings, json, random
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
warnings.filterwarnings('ignore')

import numpy as np, pandas as pd, torch
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB x{torch.cuda.device_count()}')
print('Cell 1 complete! Ready to continue without restarting ✅')


In [ ]:
# ── CELL 2: Config ───────────────────────────────────────────────────────────
# HACKATHON MODE — tune these before running

HF_TOKEN     = "YOUR_HF_TOKEN"  # expires in ~10h, fine
HF_USERNAME  = "Mushfiqul-Alam-17"
MODEL_NAME   = "medha-explainer-v1"
BASE_MODEL   = "Qwen/Qwen2.5-3B-Instruct"

# ── Training hyperparameters (optimized for 1,158 examples)
PRIORITY_CUTOFF = 228   # First 228 (38 questions * 6 states) are perfectly proofread
                        # These will ALWAYS be in training, never in validation

EPOCHS       = 6        # Was 15 — that's overfit territory for this dataset size
BATCH_SIZE   = 2
GRAD_ACCUM   = 8        # Effective batch = 16
LR           = 1e-4     # Was 2e-4 — lowered for 3B model stability
MAX_SEQ_LEN  = 1024      # Increased to 512 so JSON is never truncated
                        # Reducing this saves ~30% GPU memory and speeds training
SEED         = 42

# ── LoRA config
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

print("Config loaded ✅")
print(f"Priority examples (first {PRIORITY_CUTOFF}): always in training")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"Max sequence length: {MAX_SEQ_LEN} tokens")

In [ ]:
# ── CELL 3: Login ────────────────────────────────────────────────────────────
from huggingface_hub import login, HfApi

login(token=HF_TOKEN)
try:
    HF_USERNAME = HfApi().whoami(token=HF_TOKEN)["name"]
    print(f"Logged in as: {HF_USERNAME} ✅")
except Exception as e:
    print(f"Login ok. Could not auto-fetch username: {e}")
    print(f"Using: {HF_USERNAME}")

In [ ]:
# ── CELL 4: Load + Split Data (Priority-Aware) ───────────────────────────────
import glob
from collections import Counter

# Find data file
data_files = glob.glob("/kaggle/input/**/explainer_training_data.jsonl", recursive=True)
DATA_PATH  = data_files[0] if data_files else "explainer_training_data.jsonl"
print(f"Data: {DATA_PATH}")

raw_data = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            raw_data.append(json.loads(line))

print(f"Total examples loaded: {len(raw_data)}")

# ── Priority-aware split ─────────────────────────────────────────────────────
# First PRIORITY_CUTOFF examples = validated BD exam questions
# These MUST be in training. Never waste them on validation.
priority_examples = raw_data[:PRIORITY_CUTOFF]
remaining_examples = raw_data[PRIORITY_CUTOFF:]

# Shuffle only the remaining examples
random.seed(SEED)
random.shuffle(remaining_examples)

# Take 10% of remaining for validation
val_size  = max(50, int(len(remaining_examples) * 0.10))
val_raw   = remaining_examples[:val_size]
extra_train = remaining_examples[val_size:]

# Training = all priority + extra remaining
train_raw = (priority_examples * 3) + extra_train  # Heavily weight the proofread data (Oversampling)

print(f"\nSplit summary:")
print(f"  Priority (validated BD questions): {len(priority_examples)} (Oversampled 3x in training!)")
print(f"  Extra training:                   {len(extra_train)}")
print(f"  Total training:                   {len(train_raw)}")
print(f"  Validation:                       {len(val_raw)}")
print(f"  (Validation is from non-priority examples only)")

# ── Check class distribution
def get_state(example):
    for s in ['PRIORITY_FOCUS','GROWTH_AREA','TRUST_GAP','MASTERY']:
        if s in example['input']:
            return s
    return 'UNKNOWN'

train_states = Counter(get_state(ex) for ex in train_raw)
print(f"\nTraining distribution:")
for state, count in sorted(train_states.items()):
    pct = count / len(train_raw) * 100
    bar = '█' * (count // 10)
    print(f"  {state:<25} {count:>4} ({pct:.1f}%) {bar}")

# ── Quick output quality check
required_fields = ['explanation', 'why_wrong', 'memory_trick', 'textbook_ref']
bad_examples = 0
for ex in train_raw:
    try:
        out = json.loads(ex['output'])
        if any(f not in out for f in required_fields):
            bad_examples += 1
    except:
        bad_examples += 1

print(f"\nOutput quality: {len(train_raw) - bad_examples}/{len(train_raw)} valid ✅" if bad_examples == 0 \
      else f"\n⚠️ {bad_examples} examples have output issues")

In [ ]:
# ── CELL 5: Format Prompts ───────────────────────────────────────────────────
# Build all prompts ONCE before creating Dataset objects
# This avoids the double-formatting bug

def build_prompt(example):
    """
    Qwen2.5-Instruct chat format.
    System prompt varies based on whether student was correct or wrong.
    """
    is_correct = "(Correct)" in example["input"]
    state = get_state(example)

    system_messages = {
        "PRIORITY_FOCUS": (
            "তুমি MEDHA, বাংলাদেশের মেডিকেল ভর্তি পরীক্ষার একজন বিশেষজ্ঞ টিউটর। "
            "শিক্ষার্থী দ্রুত ও আত্মবিশ্বাসের সাথে ভুল উত্তর দিয়েছে — এটি সবচেয়ে বিপজ্জনক অবস্থা। "
            "ভুল ধারণাটি সরাসরি ও স্পষ্টভাবে সংশোধন করো।"
        ),
        "GROWTH_AREA": (
            "তুমি MEDHA, বাংলাদেশের মেডিকেল ভর্তি পরীক্ষার একজন সহানুভূতিশীল টিউটর। "
            "শিক্ষার্থী সময় নিয়েছে কিন্তু ভুল উত্তর দিয়েছে — এটি প্রকৃত জ্ঞানের ঘাটতি। "
            "সহজ ভাষায় ধারণাটি বুঝিয়ে দাও এবং মনে রাখার কৌশল দাও।"
        ),
        "TRUST_GAP": (
            "তুমি MEDHA, বাংলাদেশের মেডিকেল ভর্তি পরীক্ষার একজন টিউটর। "
            "শিক্ষার্থী সঠিক উত্তর দিয়েছে কিন্তু দ্বিধায় ছিল — তারা আসলে জানে কিন্তু বিশ্বাস করে না। "
            "তাদের আত্মবিশ্বাস বাড়াও এবং ধারণাটি পাকা করো।"
        ),
        "MASTERY": (
            "তুমি MEDHA, বাংলাদেশের মেডিকেল ভর্তি পরীক্ষার একজন টিউটর। "
            "শিক্ষার্থী দ্রুত ও সঠিকভাবে উত্তর দিয়েছে — এটি আয়ত্তের প্রমাণ। "
            "তাদের জ্ঞান আরও গভীর করো এবং সংশ্লিষ্ট ধারণার সাথে সংযোগ তৈরি করো।"
        ),
    }

    system_msg = system_messages.get(state, (
        "তুমি MEDHA, বাংলাদেশের মেডিকেল ভর্তি পরীক্ষার একজন টিউটর। "
        "শিক্ষার্থীকে সঠিক ও সহজবোধ্য ব্যাখ্যা দাও।"
    ))

    prompt = (
        f"<|im_start|>system\n{system_msg}<|im_end|>\n"
        f"<|im_start|>user\n{example['input'].strip()}<|im_end|>\n"
        f"<|im_start|>assistant\n{example['output'].strip()}<|im_end|>"
    )
    return {"text": prompt}


from datasets import Dataset

print("Formatting prompts...")
train_formatted = [build_prompt(ex) for ex in train_raw]
val_formatted   = [build_prompt(ex) for ex in val_raw]

train_dataset = Dataset.from_list(train_formatted)
val_dataset   = Dataset.from_list(val_formatted)

print(f"Train dataset: {len(train_dataset)} examples")
print(f"Val dataset:   {len(val_dataset)} examples")

# Verify format
sample = train_dataset[0]['text']
assert '<|im_start|>system' in sample, 'System block missing'
assert '<|im_start|>user' in sample, 'User block missing'
assert '<|im_start|>assistant' in sample, 'Assistant block missing'
assert '{' in sample and '}' in sample, 'JSON output missing'
print("\nFormat check ✅")
print(f"\nSample prompt preview:")
print(sample[:350] + "...")

In [ ]:
# ── CELL 6: Load Tokenizer + Verify Token Lengths ────────────────────────────
from transformers import AutoTokenizer

print(f"Loading tokenizer: {BASE_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # right padding for causal LM

# Check actual token lengths across ALL training data
print("\nChecking token lengths across all training examples...")
lengths = [len(tokenizer(ex['text'], add_special_tokens=False).input_ids) 
           for ex in train_formatted]

over_limit = sum(1 for l in lengths if l > MAX_SEQ_LEN)
print(f"Token length stats:")
print(f"  Min:          {min(lengths)}")
print(f"  Max:          {max(lengths)}")
print(f"  Avg:          {sum(lengths)//len(lengths)}")
print(f"  Over {MAX_SEQ_LEN}: {over_limit} examples")

if over_limit > 0:
    print(f"\n⚠️  {over_limit} examples exceed MAX_SEQ_LEN={MAX_SEQ_LEN}")
    print(f"   These will be truncated. Consider increasing MAX_SEQ_LEN.")
    # Find the actual max and suggest
    actual_max = max(lengths)
    suggested = ((actual_max // 64) + 1) * 64  # round up to next multiple of 64
    print(f"   Suggested MAX_SEQ_LEN: {suggested}")
else:
    print(f"\n✅ All examples within MAX_SEQ_LEN={MAX_SEQ_LEN}. No truncation.")

In [ ]:
# ── CELL 7: Load Model (4-bit QLoRA) ─────────────────────────────────────────
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print(f"Loading {BASE_MODEL} in 4-bit...")
print("(This downloads ~6GB — takes 3-5 minutes on Kaggle)")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)

# LoRA config — these are the key Qwen2.5 attention + MLP layers
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Verify memory
if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\nGPU memory: {used:.1f}GB used / {total:.1f}GB total")
    print(f"Available for training: {total - used:.1f}GB")

In [ ]:
# ── CELL 8: Setup Trainer (CUSTOM COLLATOR) ──────────────────────────────────
from transformers import Trainer, TrainingArguments, TrainerCallback
import torch

print('Tokenizing datasets...')
def tokenize_fn(examples):
    return tokenizer(examples['text'], truncation=True, max_length=MAX_SEQ_LEN)

tokenized_train = train_dataset.map(tokenize_fn, batched=True, remove_columns=train_dataset.column_names)
tokenized_val = val_dataset.map(tokenize_fn, batched=True, remove_columns=val_dataset.column_names)
print('Tokenization complete.')

class CustomCompletionCollator:
    def __init__(self, tokenizer, response_template):
        self.tokenizer = tokenizer
        self.response_template_ids = tokenizer.encode(response_template, add_special_tokens=False)
        self.ignore_index = -100

    def __call__(self, features):
        batch = self.tokenizer.pad(features, padding=True, return_tensors='pt')
        labels = batch['input_ids'].clone()
        template_len = len(self.response_template_ids)
        
        for i in range(len(labels)):
            seq = labels[i].tolist()
            match_idx = -1
            for j in range(len(seq) - template_len):
                if seq[j:j+template_len] == self.response_template_ids:
                    match_idx = j + template_len
                    break
            
            if match_idx != -1:
                labels[i, :match_idx] = self.ignore_index
            else:
                # SAFETY NET: Prevent NaN loss crashes if sequence was truncated
                labels[i, :-1] = self.ignore_index
                
            labels[i, batch['attention_mask'][i] == 0] = self.ignore_index
            
        batch['labels'] = labels
        return batch

collator = CustomCompletionCollator(tokenizer=tokenizer, response_template='assistant\n')

def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple): logits = logits[0]
    return logits.argmax(dim=-1)

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    shift_preds, shift_labels = preds[..., :-1], labels[..., 1:]
    mask = shift_labels != -100
    if not mask.any(): return {'accuracy': 0.0}
    return {'accuracy': round(float(((shift_preds == shift_labels) & mask).sum() / mask.sum()), 4)}

class MedhaProgressCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'loss' in logs:
            print(f'[Step {state.global_step:04d} | Epoch {logs.get("epoch",0):.2f}] Loss: {logs["loss"]:.4f}')
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics:
            print(f'\nEPOCH DONE | Val Loss: {metrics.get("eval_loss",0):.4f} | Acc: {metrics.get("eval_accuracy",0):.2%}\n')

training_args = TrainingArguments(
    output_dir='./qwen-medha-output', 
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE, 
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR, 
    fp16=True, 
    eval_strategy='epoch', 
    save_strategy='epoch',
    load_best_model_at_end=True, 
    metric_for_best_model='eval_loss',
    greater_is_better=False, 
    save_total_limit=3, 
    logging_steps=5,
    warmup_steps=10, 
    lr_scheduler_type='cosine', 
    seed=SEED,
    report_to='none', 
    gradient_checkpointing=True, 
    optim='paged_adamw_8bit',
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=tokenized_train, eval_dataset=tokenized_val,
    data_collator=collator, callbacks=[MedhaProgressCallback()], 
    compute_metrics=compute_metrics, preprocess_logits_for_metrics=preprocess_logits_for_metrics,
)
print("Trainer ready!")


In [ ]:
# ── CELL 9: TRAIN ────────────────────────────────────────────────────────────
print("=" * 60)
print("STARTING MEDHA EXPLAINER TRAINING")
print(f"Model:        {BASE_MODEL}")
print(f"Method:       QLoRA (4-bit) — r={LORA_R}")
print(f"Train size:   {len(train_dataset)} (incl. {PRIORITY_CUTOFF} validated BD questions)")
print(f"Val size:     {len(val_dataset)}")
print(f"Epochs:       {EPOCHS}")
print(f"Effective BS: {BATCH_SIZE * GRAD_ACCUM}")
print(f"LR:           {LR}")
print(f"Max seq len:  {MAX_SEQ_LEN}")
print("=" * 60 + "\n")

train_result = None
# 🛠️ FIX: Patch state_dict to prevent bitsandbytes CUDA illegal memory access on dual GPU
old_state_dict = trainer.model.state_dict
trainer.model.state_dict = lambda *args, **kwargs: {k: v for k, v in trainer.model.named_parameters() if v.requires_grad}

try:
    train_result = trainer.train()
    runtime_mins = train_result.metrics['train_runtime'] / 60
    print(f"\n✅ Training complete! Runtime: {runtime_mins:.1f} minutes")
    print(f"   Final train loss: {train_result.metrics['train_loss']:.4f}")
except KeyboardInterrupt:
    print("\n⏸  Training interrupted by user. Saving current state...")
except Exception as e:
    print(f"\n⚠️  Training error: {e}")
    print("   Attempting to save current model state...")

# Always save regardless of how training ended
print("\nSaving intermediate checkpoint...")
trainer.save_model("./qwen-medha-checkpoint")
tokenizer.save_pretrained("./qwen-medha-checkpoint")
print("Checkpoint saved to ./qwen-medha-checkpoint ✅")

# Restore state_dict just in case
trainer.model.state_dict = old_state_dict

In [ ]:
# ── CELL 10: Inference Test (BEFORE merging/pushing) ─────────────────────────
# Tests 3 different behavioral states to verify the model learned correctly

print("=" * 60)
print("INFERENCE TESTS")
print("=" * 60)

test_cases = [
    {
        "desc": "PRIORITY_FOCUS — fast + confident + wrong (most important case)",
        "input": (
            "Question: কোষ বিভাজনের সময় কোষপ্লেট তৈরিতে সাহায্য করে কোন অঙ্গাণু?\n"
            "Student answered: রাইবোসোম (Wrong)\n"
            "Correct answer: গলগি বস্তু\n"
            "Behavioral state: PRIORITY_FOCUS\n"
            "Chapter: কোষ ও কোষ অঙ্গাণু"
        )
    },
    {
        "desc": "TRUST_GAP — correct but hesitant",
        "input": (
            "Question: মাইটোসিসের কোন দশায় ক্রোমাটিড আলাদা হয়?\n"
            "Student answered: অ্যানাফেজ (Correct)\n"
            "Correct answer: অ্যানাফেজ\n"
            "Behavioral state: TRUST_GAP\n"
            "Chapter: কোষ বিভাজন"
        )
    },
    {
        "desc": "GROWTH_AREA — slow + wrong + guessing",
        "input": (
            "Question: কোন রক্তের গ্রুপকে সার্বজনীন দাতা বলা হয়?\n"
            "Student answered: AB (Wrong)\n"
            "Correct answer: O\n"
            "Behavioral state: GROWTH_AREA\n"
            "Chapter: রোগ প্রতিরোধ ও রক্তের গ্রুপ"
        )
    },
]

all_passed = True

for i, tc in enumerate(test_cases, 1):
    print(f"\nTest {i}: {tc['desc']}")
    print("-" * 50)
    
    # Build the prompt
    fake_example = {"input": tc["input"], "output": "{}"}
    prompt_full = build_prompt(fake_example)["text"]
    # Strip assistant turn — model will generate it
    input_text = prompt_full.split("<|im_start|>assistant\n")[0] + "<|im_start|>assistant\n"
    
    # Tokenize and generate
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, 
                       max_length=MAX_SEQ_LEN).to(model.device)
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=350,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.encode("<|im_end|>", add_special_tokens=False)[0],
        )
    
    # Extract only the generated part
    input_len = inputs['input_ids'].shape[1]
    generated = tokenizer.decode(output_ids[0][input_len:], skip_special_tokens=False)
    generated = generated.replace("<|im_end|>", "").replace("<|endoftext|>", "").strip()
    
    print(f"Generated output:")
    print(generated)
    
    # Validate
    try:
        # Handle case where model outputs text before/after JSON
        json_start = generated.find('{')
        json_end   = generated.rfind('}') + 1
        if json_start >= 0 and json_end > json_start:
            parsed = json.loads(generated[json_start:json_end])
            required = ["explanation", "why_wrong", "memory_trick", "textbook_ref"]
            missing  = [f for f in required if f not in parsed]
            if not missing:
                print(f"\n✅ Valid JSON | All fields present")
                # Check Bengali is present
                has_bengali = any('\u0980' <= c <= '\u09FF' for c in str(parsed))
                if has_bengali:
                    print(f"✅ Bengali text present")
                else:
                    print(f"⚠️  No Bengali detected — check system prompt")
            else:
                print(f"⚠️  Valid JSON but missing: {missing}")
                all_passed = False
        else:
            print("⚠️  No JSON found in output")
            all_passed = False
    except json.JSONDecodeError as e:
        print(f"❌ Invalid JSON: {e}")
        print(f"   Raw output: {generated[:200]}")
        all_passed = False

print(f"\n{'=' * 60}")
if all_passed:
    print("✅ ALL INFERENCE TESTS PASSED — model is working")
else:
    print("⚠️  SOME TESTS FAILED — model may need review")
    print("   Common causes: too few epochs, truncation issues")
    print("   The model will still be pushed — check output in production")

In [ ]:
# ── CELL 11: Merge + Push to HuggingFace ─────────────────────────────────────
# Single merge — no double-merge crash
# This is the ONLY place we call merge_and_unload()

print("Pushing LoRA adapter to HuggingFace...")
print("(Pushing adapter directly prevents multi-GPU merge crashes and is much faster)")

try:

    repo_id = f"{HF_USERNAME}/{MODEL_NAME}"
    print(f"\nPushing to: https://huggingface.co/{repo_id}")
    print("(This uploads ~100MB — takes <1 minute)")

    model.push_to_hub(repo_id, private=True, token=HF_TOKEN)
    tokenizer.push_to_hub(repo_id, private=True, token=HF_TOKEN)

    # Save training metadata
    metadata = {
        "model_name": MODEL_NAME,
        "base_model": BASE_MODEL,
        "training_examples": len(train_dataset),
        "val_examples": len(val_dataset),
        "priority_examples": PRIORITY_CUTOFF,
        "epochs": EPOCHS,
        "final_train_loss": train_result.metrics.get('train_loss') if train_result else "N/A",
        "lora_r": LORA_R,
        "output_format": "JSON with fields: explanation, why_wrong, memory_trick, textbook_ref",
        "behavioral_states": ["MASTERY", "PRIORITY_FOCUS", "TRUST_GAP", "GROWTH_AREA"],
        "language": "Bengali (primary) + English",
        "use_case": "MEDHA — BD medical admission exam behavioral study notes"
    }

    import json as _json
    with open("medha_explainer_metadata.json", "w") as f:
        _json.dump(metadata, f, indent=2, ensure_ascii=False)

    from huggingface_hub import upload_file
    upload_file(
        path_or_fileobj="medha_explainer_metadata.json",
        path_in_repo="medha_metadata.json",
        repo_id=repo_id,
        token=HF_TOKEN
    )

    print(f"\n✅ Model successfully pushed to HuggingFace!")
    print(f"   URL: https://huggingface.co/{repo_id}")
    print(f"\n   Add to your backend .env:")
    print(f"   HF_EXPLAINER_ID={repo_id}")
    print(f"   HF_TOKEN=<your production token>")

except Exception as e:
    print(f"\n❌ Push failed: {e}")
    print("\nFallback: saving adapter locally to ./qwen-medha-final/")
    try:
        model.save_pretrained("./qwen-medha-final")
        tokenizer.save_pretrained("./qwen-medha-final")
        print("✅ Saved locally. Download from Kaggle output files.")
    except Exception as e2:
        print(f"Local save also failed: {e2}")
        print("Model is still in memory — try pushing again manually.")

In [ ]:
# ── CELL 12: Backend Integration Guide ───────────────────────────────────────
print("""
=============================================================
HOW TO USE THIS MODEL IN YOUR FASTAPI BACKEND
=============================================================

OPTION A: HuggingFace Inference API (free, for demo/hackathon)
─────────────────────────────────────────────────────────────
import requests

HF_URL = f"https://api-inference.huggingface.co/models/{repo_id}"
HEADERS = {"Authorization": f"Bearer {HF_TOKEN}"}

def generate_study_note(question, wrong_answer, correct_answer, state, chapter):
    prompt = (
        f"<|im_start|>system\\n"
        f"তুমি MEDHA, বাংলাদেশের মেডিকেল ভর্তি পরীক্ষার একজন টিউটর।<|im_end|>\\n"
        f"<|im_start|>user\\n"
        f"Question: {question}\\n"
        f"Student answered: {wrong_answer} (Wrong)\\n"
        f"Correct answer: {correct_answer}\\n"
        f"Behavioral state: {state}\\n"
        f"Chapter: {chapter}<|im_end|>\\n"
        f"<|im_start|>assistant\\n"
    )
    response = requests.post(
        HF_URL,
        headers=HEADERS,
        json={"inputs": prompt, "parameters": {"max_new_tokens": 300}}
    )
    return response.json()

OPTION B: Local inference (faster, for production)
─────────────────────────────────────────────────────────────
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-3B-Instruct", torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(base, f"{repo_id}")
tokenizer = AutoTokenizer.from_pretrained(f"{repo_id}")
# Generate as usual with model.generate(...)

=============================================================
""")